# NDScan Parameter Structure Exploration

This notebook comprehensively explores the structure of NDScan experiment parameters as exposed through ARTIQ's experiment list API.

## Overview

NDScan experiments present their interface to the ARTIQ dashboard through a special parameter called `ndscan_params`. This parameter contains a JSON/PYON-encoded data structure that describes:

1. **instances** - Hierarchical organization of parameters by fragment
2. **schemata** - Complete schema definitions for each parameter
3. **always_shown** - Parameters that should always be visible in the UI
4. **overrides** - User-specified parameter value overrides
5. **scan** - Scan configuration (axes, repeats, mode)

In [1]:
import json
from sipyco import pyon
from collections import Counter, defaultdict
import pprint

# Load the experiment list
file = "explist_debug.json"
explist = json.load(open(file, "r"))
experiments = explist["experiments"]

print(f"Total experiments: {len(experiments)}")
print(f"Experiments with ndscan_params: {sum(1 for e in experiments if 'ndscan_params' in e.get('arginfo', {}))}")
print(
    f"Experiments without ndscan_params: {sum(1 for e in experiments if 'ndscan_params' not in e.get('arginfo', {}))}"
)

Total experiments: 161
Experiments with ndscan_params: 125
Experiments without ndscan_params: 36


## 1. Top-Level Structure of ndscan_params

Every NDScan experiment has an `ndscan_params` entry in its `arginfo`. Let's examine the structure.

In [2]:
# Get all ndscan experiments
ndscan_experiments = [e for e in experiments if "ndscan_params" in e.get("arginfo", {})]

# Parse all ndscan_params
parsed_experiments = []
for exp in ndscan_experiments:
    ndscan_raw = exp["arginfo"]["ndscan_params"]
    # ndscan_params is a list: [arginfo_dict, group, tooltip]
    if ndscan_raw[0] is not None:
        parsed = pyon.decode(ndscan_raw[0]["default"])
        parsed_experiments.append({"name": exp["name"], "class_name": exp["class_name"], "ndscan": parsed})

print(f"Successfully parsed {len(parsed_experiments)} experiments")

# Show top-level keys present in all ndscan structures
all_keys = set()
for exp in parsed_experiments:
    all_keys.update(exp["ndscan"].keys())
print(f"\nTop-level keys in ndscan_params: {sorted(all_keys)}")

Successfully parsed 125 experiments

Top-level keys in ndscan_params: ['always_shown', 'instances', 'overrides', 'scan', 'schemata']


In [3]:
# Examine one experiment in detail
sample_exp = parsed_experiments[0]
print(f"Experiment: {sample_exp['name']}")
print(f"Class: {sample_exp['class_name']}")
print("\nTop-level structure:")
for key, value in sample_exp["ndscan"].items():
    if isinstance(value, dict):
        print(f"  {key}: dict with {len(value)} entries")
    elif isinstance(value, list):
        print(f"  {key}: list with {len(value)} entries")
    else:
        print(f"  {key}: {type(value).__name__} = {value}")

Experiment: injected_diodes/AllRelockers
Class: AllRelockers

Top-level structure:
  instances: dict with 5 entries
  schemata: dict with 54 entries
  always_shown: list with 6 entries
  overrides: dict with 0 entries
  scan: dict with 4 entries


## 2. The `instances` Structure

The `instances` dict maps fragment paths to lists of parameter FQNs. The empty string `""` represents the root fragment.

In [4]:
# Analyze instance structures across all experiments
instance_stats = []
for exp in parsed_experiments:
    instances = exp["ndscan"].get("instances", {})
    instance_stats.append(
        {
            "name": exp["name"],
            "num_fragments": len(instances),
            "root_params": len(instances.get("", [])),
            "total_params": sum(len(v) for v in instances.values()),
        }
    )

# Show experiments with most fragments (complex hierarchies)
sorted_by_fragments = sorted(instance_stats, key=lambda x: x["num_fragments"], reverse=True)[:10]
print("Top 10 experiments by fragment count:")
for stat in sorted_by_fragments:
    print(f"  {stat['name']}: {stat['num_fragments']} fragments, {stat['total_params']} params")

Top 10 experiments by fragment count:
  clock_spectroscopy/Clock spectroscopy from dropped single XODT with evaporation, shaped shelving and clearout: 198 fragments, 413 params
  clock_spectroscopy/Shaped clock spectroscopy from dropped, velocity-sliced and evaporated single XODT: 198 fragments, 413 params
  clock_spectroscopy/Clock spectroscopy from dropped single XODT with evaporation, shelving and clearout: 196 fragments, 412 params
  clock_spectroscopy/Down beam clock spectroscopy from dropped single XODT with evaporation, shelving and clearout: 196 fragments, 412 params
  clock_interferometry/Clock interferometry from dropped single XODT with evaporation, shaped shelving and clearout: 195 fragments, 409 params
  clock_spectroscopy/Clock spectroscopy from dropped single XODT with evaporation: 192 fragments, 407 params
  clock_interferometry/Clock interferometry from a double XODT with signal and noise: 192 fragments, 388 params
  clock_spectroscopy/Down beam clock spectroscopy from

In [5]:
# Show fragment hierarchy for a complex experiment
complex_exp = next(e for e in parsed_experiments if e["class_name"] == "RelockAllIJDs")
print(f"Fragment hierarchy for {complex_exp['name']}:\n")
instances = complex_exp["ndscan"]["instances"]
for fragment_path, params in sorted(instances.items()):
    indent = "  " * fragment_path.count("/")
    display_name = fragment_path if fragment_path else "(root)"
    print(f"{indent}{display_name}: {len(params)} params")

Fragment hierarchy for injected_diodes/Relock all IJDs:

(root): 8 params
frag_relocker_blue_IJD1_controller: 6 params
  frag_relocker_blue_IJD1_controller/beam_setter: 2 params
    frag_relocker_blue_IJD1_controller/beam_setter/urukul_init: 0 params
  frag_relocker_blue_IJD1_controller/frag_blue_IJD1_relocker: 14 params
  frag_relocker_blue_IJD1_controller/frag_koheron_blue_IJD1_controller: 4 params
    frag_relocker_blue_IJD1_controller/frag_koheron_blue_IJD1_controller/adc_reader: 1 params
frag_relocker_blue_IJD2_controller: 6 params
  frag_relocker_blue_IJD2_controller/beam_setter: 0 params
    frag_relocker_blue_IJD2_controller/beam_setter/urukul_init: 0 params
  frag_relocker_blue_IJD2_controller/frag_blue_IJD2_relocker: 14 params
  frag_relocker_blue_IJD2_controller/frag_koheron_blue_IJD2_controller: 4 params
    frag_relocker_blue_IJD2_controller/frag_koheron_blue_IJD2_controller/adc_reader: 1 params
frag_relocker_blue_IJD3_controller: 6 params
  frag_relocker_blue_IJD3_control

## 3. The `schemata` Structure - Parameter Types

Each parameter has a schema that defines its type, default value, and specification.

In [6]:
# Collect all unique parameter types
all_param_types = Counter()
all_spec_keys = set()

for exp in parsed_experiments:
    schemata = exp["ndscan"].get("schemata", {})
    for fqn, schema in schemata.items():
        param_type = schema.get("type", "unknown")
        all_param_types[param_type] += 1
        spec = schema.get("spec", {})
        all_spec_keys.update(spec.keys())

print("Parameter types found:")
for ptype, count in all_param_types.most_common():
    print(f"  {ptype}: {count}")

print(f"\nSpec keys found: {sorted(all_spec_keys)}")

Parameter types found:
  float: 13849
  int: 3076
  bool: 691
  enum: 1
  string: 1

Spec keys found: ['is_scannable', 'max', 'members', 'min', 'scale', 'step', 'unit']


In [7]:
# Collect example schemas for each type
type_examples = {}
for exp in parsed_experiments:
    schemata = exp["ndscan"].get("schemata", {})
    for fqn, schema in schemata.items():
        param_type = schema.get("type", "unknown")
        if param_type not in type_examples:
            type_examples[param_type] = schema

print("Example schemas for each type:\n")
for ptype, example in sorted(type_examples.items()):
    print(f"=== {ptype.upper()} ===")
    pprint.pprint(example)
    print()

Example schemas for each type:

=== BOOL ===
{'default': 'True',
 'description': 'blue_IJD1_relocker enabled',
 'fqn': 'relocker_board.AllRelockersFrag.blue_IJD1_relocker_enabled',
 'spec': {'is_scannable': True},
 'type': 'bool'}

=== ENUM ===
{'default': "'up'",
 'description': 'Spectroscopy beam',
 'fqn': '689_spectroscopy_from_xxodt.RedSpectroscopyFromXXODTFrag.spectroscopy_beam',
 'spec': {'is_scannable': True,
          'members': {'sigmaminus': 'red_mot_sigmaminus',
                      'sigmaplus': 'red_mot_sigmaplus',
                      'up': 'red_up'}},
 'type': 'enum'}

=== FLOAT ===
{'default': '-0.5333333333333333',
 'description': 'v_min',
 'fqn': 'relocker_board.AllRelockersFrag.build_fragment.<locals>._RelockerChannelFrag_blue_IJD1_relocker.v_min',
 'spec': {'is_scannable': True,
          'max': 4.0,
          'min': -4.0,
          'scale': 1,
          'step': 0.01,
          'unit': 'Volts'},
 'type': 'float'}

=== INT ===
{'default': '100',
 'description': 'n s

## 4. Float Parameter Variations

Float parameters have the most variation in their spec fields. Let's explore all variations.

In [8]:
# Collect all float parameter specs
float_specs = []
for exp in parsed_experiments:
    schemata = exp["ndscan"].get("schemata", {})
    for fqn, schema in schemata.items():
        if schema.get("type") == "float":
            float_specs.append(
                {
                    "fqn": fqn,
                    "description": schema.get("description", ""),
                    "default": schema.get("default"),
                    "spec": schema.get("spec", {}),
                }
            )

print(f"Total float parameters: {len(float_specs)}")

# Find unique combinations of spec keys
spec_key_combos = Counter()
for fs in float_specs:
    keys = tuple(sorted(fs["spec"].keys()))
    spec_key_combos[keys] += 1

print("\nFloat spec key combinations:")
for combo, count in spec_key_combos.most_common():
    print(f"  {combo}: {count}")

Total float parameters: 13849

Float spec key combinations:
  ('is_scannable', 'min', 'scale', 'step', 'unit'): 3257
  ('is_scannable', 'scale', 'step'): 2936
  ('is_scannable', 'max', 'scale', 'step'): 2137
  ('is_scannable', 'scale', 'step', 'unit'): 1821
  ('is_scannable', 'max', 'min', 'scale', 'step', 'unit'): 1533
  ('is_scannable', 'min', 'scale', 'step'): 1236
  ('is_scannable', 'max', 'min', 'scale', 'step'): 929


In [9]:
# Show examples of floats with different units
units_found = defaultdict(list)
for fs in float_specs:
    unit = fs["spec"].get("unit", "(no unit)")
    units_found[unit].append(fs)

print(f"Unique units: {len(units_found)}")
print("\nExamples by unit:")
for unit, params in sorted(units_found.items())[:15]:
    example = params[0]
    print(f"\n  {unit} ({len(params)} params):")
    print(f"    Example: {example['description']}")
    print(f"    Default: {example['default']}, Scale: {example['spec'].get('scale', 1)}")

Unique units: 17

Examples by unit:

  (no unit) (7238 params):
    Example: window fraction
    Default: 0.2, Scale: 1

  A (1352 params):
    Example: Current to set
    Default: 0.0, Scale: 1.0

  GHz (6 params):
    Example: Tolerance for correct mode detection
    Default: 10000000000.0, Scale: 1000000000.0

  Hz (12 params):
    Example: frequency
    Default: 1, Scale: 1.0

  MHz (1934 params):
    Example: Frequency for red_doublepass_injection
    Default: 366900000.0, Scale: 1000000.0

  Ohms (2 params):
    Example: Temperature
    Default: 8800, Scale: 1

  THz (1 params):
    Example: Reference frequency
    Default: 0.0, Scale: 1000000000000.0

  V (914 params):
    Example: voltage step up on relock
    Default: 0.8, Scale: 1.0

  Volts (26 params):
    Example: v_min
    Default: -0.5333333333333333, Scale: 1

  dB (6 params):
    Example: Output attenuation
    Default: 30.0, Scale: 1.0

  kHz (308 params):
    Example: Detuning of the 689 stir beam during xodt loading

## 5. Int Parameter Variations

In [10]:
# Collect all int parameter specs
int_specs = []
for exp in parsed_experiments:
    schemata = exp["ndscan"].get("schemata", {})
    for fqn, schema in schemata.items():
        if schema.get("type") == "int":
            int_specs.append(
                {
                    "fqn": fqn,
                    "description": schema.get("description", ""),
                    "default": schema.get("default"),
                    "spec": schema.get("spec", {}),
                }
            )

print(f"Total int parameters: {len(int_specs)}")

# Show examples
print("\nExample int parameters:")
for spec in int_specs[:5]:
    print(f"  {spec['description']}: default={spec['default']}, spec={spec['spec']}")

Total int parameters: 3076

Example int parameters:
  n steps: default=100, spec={'is_scannable': True, 'scale': 1, 'min': 10, 'max': 128}
  Smoothing factor for averaging. Bigger = more smoothing: default=10000, spec={'is_scannable': True, 'scale': 1, 'min': 0, 'max': 2147483648}
  n steps: default=100, spec={'is_scannable': True, 'scale': 1, 'min': 10, 'max': 128}
  Smoothing factor for averaging. Bigger = more smoothing: default=10000, spec={'is_scannable': True, 'scale': 1, 'min': 0, 'max': 2147483648}
  n steps: default=100, spec={'is_scannable': True, 'scale': 1, 'min': 10, 'max': 128}


## 6. Bool Parameter Structure

In [11]:
# Collect all bool parameter specs
bool_specs = []
for exp in parsed_experiments:
    schemata = exp["ndscan"].get("schemata", {})
    for fqn, schema in schemata.items():
        if schema.get("type") == "bool":
            bool_specs.append(
                {
                    "fqn": fqn,
                    "description": schema.get("description", ""),
                    "default": schema.get("default"),
                    "spec": schema.get("spec", {}),
                }
            )

print(f"Total bool parameters: {len(bool_specs)}")

# Check if all bools have the same spec structure
bool_spec_combos = Counter()
for bs in bool_specs:
    keys = tuple(sorted(bs["spec"].keys()))
    bool_spec_combos[keys] += 1

print("\nBool spec key combinations:")
for combo, count in bool_spec_combos.most_common():
    print(f"  {combo}: {count}")

# Show default value distribution
true_count = sum(1 for bs in bool_specs if bs["default"] == "True")
false_count = sum(1 for bs in bool_specs if bs["default"] == "False")
print(f"\nDefault True: {true_count}, Default False: {false_count}")

Total bool parameters: 691

Bool spec key combinations:
  ('is_scannable',): 691

Default True: 327, Default False: 364


## 7. The `always_shown` Structure

This defines which parameters should always be visible in the dashboard UI.

In [12]:
# Analyze always_shown structures
always_shown_stats = []
for exp in parsed_experiments:
    always_shown = exp["ndscan"].get("always_shown", [])
    always_shown_stats.append(
        {
            "name": exp["name"],
            "count": len(always_shown),
            "entries": always_shown[:3],  # First 3 entries
        }
    )

# Show distribution
counts = Counter(s["count"] for s in always_shown_stats)
print("Distribution of always_shown count:")
for count, num_exps in sorted(counts.items())[:10]:
    print(f"  {count} params always shown: {num_exps} experiments")

Distribution of always_shown count:
  0 params always shown: 28 experiments
  1 params always shown: 9 experiments
  2 params always shown: 4 experiments
  3 params always shown: 8 experiments
  4 params always shown: 7 experiments
  5 params always shown: 6 experiments
  6 params always shown: 4 experiments
  7 params always shown: 2 experiments
  8 params always shown: 1 experiments
  9 params always shown: 5 experiments


In [13]:
# Examine the format of always_shown entries
for exp in parsed_experiments:
    always_shown = exp["ndscan"].get("always_shown", [])
    if always_shown:
        print("Format of always_shown entries:")
        pprint.pprint(always_shown[:3])
        break

# The __jsonclass__ format indicates a tuple encoded in JSON
# Format: {"__jsonclass__": ["tuple", [[fqn, fragment_path]]]}

Format of always_shown entries:
[('relocker_board.AllRelockersFrag.blue_IJD1_relocker_enabled', ''),
 ('relocker_board.AllRelockersFrag.blue_IJD2_relocker_enabled', ''),
 ('relocker_board.AllRelockersFrag.blue_IJD3_relocker_enabled', '')]


## 8. The `scan` Structure

Defines scan configuration - which parameters to scan over and how.

In [14]:
# Collect all scan configurations
scan_configs = []
for exp in parsed_experiments:
    scan = exp["ndscan"].get("scan", {})
    scan_configs.append({"name": exp["name"], "scan": scan})

# Show the scan structure
print("Scan structure keys:")
all_scan_keys = set()
for sc in scan_configs:
    all_scan_keys.update(sc["scan"].keys())
print(f"  {sorted(all_scan_keys)}")

# Show example
print("\nExample scan configuration:")
pprint.pprint(scan_configs[0]["scan"])

Scan structure keys:
  ['axes', 'no_axes_mode', 'num_repeats', 'randomise_order_globally']

Example scan configuration:
{'axes': [],
 'no_axes_mode': 'single',
 'num_repeats': 1,
 'randomise_order_globally': False}


In [15]:
# Analyze no_axes_mode values
no_axes_modes = Counter()
for sc in scan_configs:
    mode = sc["scan"].get("no_axes_mode", "unknown")
    no_axes_modes[mode] += 1

print("no_axes_mode values:")
for mode, count in no_axes_modes.most_common():
    print(f"  {mode}: {count}")

# Check for any experiments with pre-configured axes
exps_with_axes = [sc for sc in scan_configs if sc["scan"].get("axes", [])]
print(f"\nExperiments with pre-configured scan axes: {len(exps_with_axes)}")

no_axes_mode values:
  single: 125

Experiments with pre-configured scan axes: 0


## 9. The `overrides` Structure

Holds user-specified parameter value overrides.

In [16]:
# Check for any experiments with pre-set overrides
exps_with_overrides = []
for exp in parsed_experiments:
    overrides = exp["ndscan"].get("overrides", {})
    if overrides:
        exps_with_overrides.append({"name": exp["name"], "overrides": overrides})

print(f"Experiments with overrides: {len(exps_with_overrides)}")
if exps_with_overrides:
    print("\nExample overrides:")
    pprint.pprint(exps_with_overrides[0])

Experiments with overrides: 0


## 10. Non-NDScan Parameters (arginfo)

Some experiments have additional parameters outside of ndscan_params.

In [17]:
# Find experiments with both ndscan and non-ndscan params
mixed_experiments = []
for exp in ndscan_experiments:
    arginfo = exp.get("arginfo", {})
    non_ndscan_params = [k for k in arginfo.keys() if k != "ndscan_params"]
    if non_ndscan_params:
        mixed_experiments.append({"name": exp["name"], "non_ndscan_params": non_ndscan_params, "arginfo": arginfo})

print(f"Experiments with both ndscan and non-ndscan params: {len(mixed_experiments)}")
print("\nExamples:")
for exp in mixed_experiments[:5]:
    print(f"  {exp['name']}: {exp['non_ndscan_params']}")

Experiments with both ndscan and non-ndscan params: 24

Examples:
  injected_diodes/Single relocker board channel: ['channel_name']
  injected_diodes/ScanIJDRelocker: ['channel_name']
  injected_diodes/Set a Koheron CTL200 laser driver's current and measure an analog input in response: ['controller_name']
  utilities/Set an AD9910 or AD9912 DDS channel, while setting all other channels on this Urukul to their defaults: ['device_name']
  utilities/Set the current for an analog current supply: ['current_supply']


In [18]:
# Collect all non-ndscan parameter types
param_types = Counter()
for exp in mixed_experiments:
    for param_name, param_info in exp["arginfo"].items():
        if param_name != "ndscan_params" and param_info[0] is not None:
            param_types[param_info[0].get("ty", "unknown")] += 1

print("Non-ndscan parameter types:")
for ptype, count in param_types.most_common():
    print(f"  {ptype}: {count}")

Non-ndscan parameter types:
  BooleanValue: 31
  NumberValue: 21
  EnumerationValue: 16
  StringValue: 1


In [19]:
# Example of EnumerationValue (common for channel selection)
for exp in mixed_experiments:
    for param_name, param_info in exp["arginfo"].items():
        if param_name != "ndscan_params" and param_info[0] is not None:
            if param_info[0].get("ty") == "EnumerationValue":
                print(f"EnumerationValue example from {exp['name']}:")
                print(f"  Param: {param_name}")
                pprint.pprint(param_info[0])
                break
    else:
        continue
    break

EnumerationValue example from injected_diodes/Single relocker board channel:
  Param: channel_name
{'choices': ['blue_IJD1_relocker',
             'blue_IJD2_relocker',
             'blue_IJD3_relocker',
             'red_IJD1_relocker'],
 'default': 'blue_IJD1_relocker',
 'quickstyle': False,
 'ty': 'EnumerationValue'}


## Summary

Key findings about NDScan parameter structure:

### Top-Level Keys
- `instances`: Dict mapping fragment paths to parameter FQN lists
- `schemata`: Dict mapping FQNs to complete parameter schemas
- `always_shown`: List of (fqn, fragment_path) tuples for UI display
- `overrides`: Dict of user-specified value overrides (usually empty)
- `scan`: Scan configuration object

### Parameter Types
- `float`: Most common, with scale, step, min, max, unit
- `int`: Integer values with scale, min, max
- `bool`: Boolean values, all scannable

### Spec Fields
- `is_scannable`: Always true for NDScan params
- `scale`: Multiplier for display/internal conversion
- `step`: UI increment step
- `min`, `max`: Value bounds (optional)
- `unit`: Display unit string